In [3]:
!pip install google-adk google-generativeai pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 3.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires ric

In [11]:
# Create the proper ADK agent directory structure
import os
import shutil

# Clean up any existing directory
if os.path.exists('dwaste-agent'):
    shutil.rmtree('dwaste-agent')

# Create the directory structure
os.makedirs('dwaste-agent', exist_ok=True)
print("✅ Created dwaste-agent directory")

✅ Created dwaste-agent directory


In [54]:
%%writefile dwaste-agent/agent.py

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.function_tool import FunctionTool
from google.genai import types
import io
import json
import uuid
import logging
import base64
import requests
from datetime import datetime, timedelta
from typing import Dict, Any, List, Optional
from PIL import Image
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dwaste-agent")

# ---------------------- Gemini AI Client ----------------------
class GeminiClient:
    def __init__(self, api_key: str):
        genai.configure(api_key=api_key)
        # Use the correct model name that works with the API
        self.model = genai.GenerativeModel('gemini-2.5-flash-lite')
    
    def analyze_from_description(self, item_description: str, location: str = "default") -> Dict[str, Any]:
        """Analyze recycling item from text description"""
        try:
            prompt = f"""
            Analyze this waste/recycling item based on description and provide structured information:

            ITEM DESCRIPTION: {item_description}
            LOCATION: {location}

            Please provide the following information in JSON format:
            1. ITEM_IDENTIFICATION: What is this item specifically?
            2. MATERIAL_COMPOSITION: What materials is it made of?
            3. RECYCLING_CATEGORY: recyclable, compostable, hazardous, e-waste, special, trash
            4. CONFIDENCE: 0.0 to 1.0
            5. INSTRUCTIONS: Specific disposal instructions
            6. ENVIRONMENTAL_IMPACT: Brief educational message about environmental impact

            Be precise and provide actionable guidance.
            """
            
            response = self.model.generate_content(prompt)
            response_text = response.text.strip()
            
            # Clean response text
            if response_text.startswith('```json'):
                response_text = response_text[7:-3].strip()
            elif response_text.startswith('```'):
                response_text = response_text[3:-3].strip()
            
            analysis = json.loads(response_text)
            return analysis
            
        except Exception as e:
            logger.error(f"Gemini analysis failed: {e}")
            return {
                "ITEM_IDENTIFICATION": "Unknown Item",
                "MATERIAL_COMPOSITION": "Unknown Material",
                "RECYCLING_CATEGORY": "trash",
                "CONFIDENCE": 0.5,
                "INSTRUCTIONS": "Please check with local recycling guidelines for proper disposal",
                "ENVIRONMENTAL_IMPACT": "Proper waste disposal helps protect our environment and conserve resources"
            }
    
    def analyze_from_image_url(self, image_url: str, location: str = "default") -> Dict[str, Any]:
        """Analyze recycling item from image URL"""
        try:
            # Download image from URL
            response = requests.get(image_url)
            response.raise_for_status()
            
            prompt = f"""
            Analyze this waste/recycling item from the image and provide structured information:

            LOCATION: {location}

            Please provide the following information in JSON format:
            1. ITEM_IDENTIFICATION: What is this item specifically?
            2. MATERIAL_COMPOSITION: What materials is it made of?
            3. RECYCLING_CATEGORY: recyclable, compostable, hazardous, e-waste, special, trash
            4. CONFIDENCE: 0.0 to 1.0
            5. INSTRUCTIONS: Specific disposal instructions
            6. ENVIRONMENTAL_IMPACT: Brief educational message about environmental impact

            Be precise and provide actionable guidance.
            """
            
            image = Image.open(io.BytesIO(response.content))
            response = self.model.generate_content([prompt, image])
            response_text = response.text.strip()
            
            # Clean response text
            if response_text.startswith('```json'):
                response_text = response_text[7:-3].strip()
            elif response_text.startswith('```'):
                response_text = response_text[3:-3].strip()
            
            analysis = json.loads(response_text)
            return analysis
            
        except Exception as e:
            logger.error(f"Image URL analysis failed: {e}")
            return self.analyze_from_description(f"Item from image at {image_url}", location)

# ---------------------- In-Memory Database ----------------------
class InMemoryDB:
    def __init__(self):
        self.records = []
    
    def insert_record(self, record: Dict[str, Any]):
        self.records.append(record)
    
    def get_recent(self, minutes=60):
        cutoff = datetime.utcnow() - timedelta(minutes=minutes)
        return [r for r in self.records if r.get("timestamp") and r["timestamp"] >= cutoff]
    
    def all(self):
        return list(self.records)
    
    def get_category_stats(self):
        categories = ["recyclable", "compostable", "hazardous", "e-waste", "trash"]
        stats = {}
        for category in categories:
            stats[category] = len([r for r in self.records if r.get("category") == category])
        return stats

# Initialize global database
DB = InMemoryDB()

# Add test records
test_records = [
    {
        "record_id": "test_001",
        "item_identification": "Plastic Water Bottle",
        "category": "recyclable",
        "confidence": 0.95,
        "timestamp": datetime.utcnow() - timedelta(hours=2)
    },
    {
        "record_id": "test_002", 
        "item_identification": "Apple Core",
        "category": "compostable",
        "confidence": 0.98,
        "timestamp": datetime.utcnow() - timedelta(hours=1)
    }
]

for record in test_records:
    DB.insert_record(record)

# ---------------------- Agent Tools ----------------------
def analyze_recycling_item(item_description: str) -> dict:
    """
    Analyze a recycling item from text description
    
    Args:
        item_description: Text description of the recycling item
    """
    try:
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("GOOGLE_API_KEY")
        
        gemini_client = GeminiClient(api_key)
        analysis = gemini_client.analyze_from_description(item_description, "web_ui")
        
        record_id = uuid.uuid4().hex
        record = {
            "record_id": record_id,
            "item_identification": analysis.get("ITEM_IDENTIFICATION", "Unknown Item"),
            "material_composition": analysis.get("MATERIAL_COMPOSITION", "Unknown Material"),
            "category": analysis.get("RECYCLING_CATEGORY", "trash").lower(),
            "confidence": analysis.get("CONFIDENCE", 0.5),
            "instructions": analysis.get("INSTRUCTIONS", "Check local recycling guidelines"),
            "environmental_impact": analysis.get("ENVIRONMENTAL_IMPACT", "Proper disposal helps protect our environment"),
            "timestamp": datetime.utcnow(),
            "source": "text_description",
            "input": item_description[:100]  # Store first 100 chars of input
        }
        
        DB.insert_record(record)
        
        return {
            "status": "success",
            "analysis": analysis,
            "record_id": record_id
        }
        
    except Exception as e:
        logger.error(f"Error in analyze_recycling_item tool: {e}")
        return {
            "status": "error",
            "error_message": f"Failed to analyze item: {str(e)}"
        }

def get_recycling_stats() -> dict:
    """Get recycling statistics and analytics."""
    try:
        total_records = len(DB.all())
        recent_1h = len(DB.get_recent(minutes=60))
        category_stats = DB.get_category_stats()
        
        recyclable_count = category_stats.get("recyclable", 0)
        compostable_count = category_stats.get("compostable", 0)
        diverted_from_landfill = recyclable_count + compostable_count
        diversion_rate = (diverted_from_landfill / total_records * 100) if total_records > 0 else 0
        
        return {
            "status": "success",
            "stats": {
                "total_records": total_records,
                "recent_1h": recent_1h,
                "category_breakdown": category_stats,
                "environmental_impact": {
                    "diverted_from_landfill": diverted_from_landfill,
                    "diversion_rate": diversion_rate
                }
            }
        }
        
    except Exception as e:
        logger.error(f"Error in get_recycling_stats tool: {e}")
        return {
            "status": "error",
            "error_message": f"Failed to get recycling stats: {str(e)}"
        }

def get_recycling_history() -> dict:
    """Get recent recycling history."""
    try:
        records = DB.all()
        recent_records = records[-10:] if len(records) > 10 else records
        
        return {
            "status": "success",
            "total_records": len(records),
            "recent_records": recent_records
        }
        
    except Exception as e:
        logger.error(f"Error in get_recycling_history tool: {e}")
        return {
            "status": "error",
            "error_message": f"Failed to get recycling history: {str(e)}"
        }

# Create tools
analyze_tool = FunctionTool(analyze_recycling_item)
stats_tool = FunctionTool(get_recycling_stats)
history_tool = FunctionTool(get_recycling_history)

# Set names and descriptions
analyze_tool.name = "analyze_recycling_item"
analyze_tool.description = "Analyze a recycling item from text description and provide disposal instructions"

stats_tool.name = "get_recycling_stats"
stats_tool.description = "Get recycling statistics and analytics"

history_tool.name = "get_recycling_history"  
history_tool.description = "Get recent recycling history and records"

# ---------------------- Main Agent ----------------------
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

# THIS MUST BE NAMED root_agent FOR ADK TO RECOGNIZE IT
# Use the correct model that works with ADK
root_agent = LlmAgent(
    name="dwaste_recycling_assistant",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description="An AI-powered recycling assistant that analyzes waste items from text descriptions and provides disposal instructions",
    instruction="""
    You are DWaste - an intelligent recycling assistant. Your role is to help users with waste classification and recycling guidance.

    When users interact with you:

    1. **For Item Analysis**:
       - When users describe recycling items, use the analyze_recycling_item tool
       - The tool accepts text descriptions of items
       - Provide detailed analysis including:
         * Item identification
         * Material composition  
         * Recycling category (recyclable, compostable, hazardous, e-waste, trash)
         * Specific disposal instructions
         * Environmental impact information
       - Be encouraging and educational

    2. **For Analytics Requests**:
       - When users ask for "stats", "analytics", or "status", use the get_recycling_stats tool
       - Present statistics in a clear, informative way
       - Highlight environmental impact and recycling progress

    3. **For History Requests**:
       - When users ask for "records", "history", or "past items", use the get_recycling_history tool
       - Show recent recycling activity and patterns

    4. **General Interaction**:
       - Be friendly, helpful, and environmentally conscious
       - Encourage proper recycling habits
       - Provide educational information about waste management
       - Use emojis to make responses engaging (♻️🍂⚠️🔋🗑️)

    IMPORTANT: Only use the available tools:
    - analyze_recycling_item: For analyzing recycling items from text descriptions
    - get_recycling_stats: For getting statistics and analytics
    - get_recycling_history: For viewing recycling history

    Always respond in a conversational, helpful tone and focus on empowering users to make better recycling decisions.
    """,
    tools=[analyze_tool, stats_tool, history_tool]
)

print("✅ DWaste agent created successfully!")

Overwriting dwaste-agent/agent.py


In [55]:
# Verify the folder structure was created correctly
print("📁 Checking agent structure...")
print("Files in dwaste-agent directory:")
!ls -la dwaste-agent/

print("\n🔧 Testing agent import...")
try:
    # Import directly from the agent.py file
    import importlib.util
    import sys
    
    # Add the current directory to Python path
    sys.path.insert(0, '.')
    
    # Load the agent module directly
    spec = importlib.util.spec_from_file_location("agent", "dwaste-agent/agent.py")
    agent_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(agent_module)
    
    # Access the root_agent
    root_agent = agent_module.root_agent
    
    print(f"✅ Agent loaded successfully!")
    print(f"🤖 Agent name: {root_agent.name}")
    print(f"🛠️ Tools available: {len(root_agent.tools)}")
    print(f"📝 Description: {root_agent.description}")
    
    # Test if tools work
    print(f"\n🔧 Testing tools...")
    stats_result = agent_module.get_recycling_stats()
    print(f"📊 Stats tool test: {stats_result['status']}")
    if stats_result['status'] == 'success':
        print(f"   Total records: {stats_result['stats']['total_records']}")
        print(f"   Category breakdown: {stats_result['stats']['category_breakdown']}")
    
    # Test the analysis tool
    print(f"\n🔧 Testing analysis tool...")
    analysis_result = agent_module.analyze_recycling_item("banana peel")
    print(f"📝 Analysis tool test: {analysis_result['status']}")
    if analysis_result['status'] == 'success':
        print(f"   Item: {analysis_result['analysis'].get('ITEM_IDENTIFICATION', 'Unknown')}")
        print(f"   Category: {analysis_result['analysis'].get('RECYCLING_CATEGORY', 'Unknown')}")
    
except Exception as e:
    print(f"❌ Error loading agent: {e}")
    import traceback
    traceback.print_exc()

📁 Checking agent structure...
Files in dwaste-agent directory:
total 36
drwxr-xr-x 3 root root  4096 Nov 24 23:42 .
drwxr-xr-x 4 root root  4096 Nov 24 23:34 ..
-rw-r--r-- 1 root root 12507 Nov 25 00:18 agent.py
-rw-r--r-- 1 root root    54 Nov 24 23:44 .env
-rw-r--r-- 1 root root    49 Nov 24 23:44 __init__.py
drwxr-xr-x 2 root root  4096 Nov 25 00:17 __pycache__

🔧 Testing agent import...
✅ DWaste agent created successfully!
✅ Agent loaded successfully!
🤖 Agent name: dwaste_recycling_assistant
🛠️ Tools available: 3
📝 Description: An AI-powered recycling assistant that analyzes waste items from text descriptions and provides disposal instructions

🔧 Testing tools...
📊 Stats tool test: success
   Total records: 2
   Category breakdown: {'recyclable': 1, 'compostable': 1, 'hazardous': 0, 'e-waste': 0, 'trash': 0}

🔧 Testing analysis tool...
📝 Analysis tool test: success
   Item: Banana Peel
   Category: compostable


In [56]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))
    return url_prefix

print("✅ Helper functions defined.")

✅ Helper functions defined.


In [57]:
url_prefix = get_adk_proxy_url()
print(f"🔗 Proxy URL prefix: {url_prefix}")

🔗 Proxy URL prefix: /k/281536754/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidHlwIjoiSldUIn0..6l4z5W8YOIzcXnZ-8P9N8g.bZofww017sTx8Rd9-Num9eSoPDi_4fL7WJXItxR9KtGypkKWlt5F4ag5t4AOWAOOeqOrpwQRTGWXnfkvWtnw84KjOzjtDj9c3_k4L1HEEg2u5Bt06u67nKMTGamoVF4EEkRBvjOGAp6EKC37tgw1dCUrOXSJCcusq0HdkggIOTHRAT-6mT7OqBkqWmQDqyvTVwxPG-u9rNEpn0An1zHkv4cScL4DBmUALjG4T-c61RDZEXf_qy2Sn4gAiNxFL8N8.7mlZNbLg8cCb70dwFpRdyg/proxy/proxy/8000


In [58]:
# ---------------------- Start ADK Web UI ----------------------
print("🚀 Starting ADK Web UI...")
print("⏳ This may take a few seconds...")
print("📁 Current directory:", os.getcwd())
print("📂 Agent directory exists:", os.path.exists('dwaste-agent'))

# Start the ADK web interface
!adk web --log_level DEBUG --url_prefix {url_prefix}

🚀 Starting ADK Web UI...
⏳ This may take a few seconds...
📁 Current directory: /kaggle/working
📂 Agent directory exists: True
/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [204]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                              